# อุบัติเหตุทางถนนช่วงสงกรานต์และปีใหม่ — walkthrough

Notebook นี้เดินผ่าน pipeline ทีละขั้น (โค้ดจริงอยู่ในแพ็กเกจ `roadsafety/`) เพื่อให้เห็นว่าข้อมูลดิบไม่เรียบร้อยแค่ไหนและแก้อย่างไร

1. โหลดไฟล์ดิบจาก `data/raw/`
2. สำรวจปัญหาของข้อมูล
3. ทำความสะอาด + บันทึก cleaning log
4. วิเคราะห์ตามเวลา (ช่วงเทศกาล / วันที่ของเทศกาล / ชั่วโมง)
5. วิเคราะห์ตามพื้นที่ (จังหวัด / ภาค)
6. สาเหตุและประเภทรถ
7. สรุป insight และข้อเสนอแนะเชิงนโยบาย → `reports/REPORT.md`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from roadsafety import analysis, charts
from roadsafety.clean import CleaningLog, clean, read_raw, standardize_columns
from roadsafety.features import add_features

RAW = pathlib.Path("../data/raw")
files = sorted(p for p in RAW.iterdir() if p.suffix.lower() in {".csv", ".xls", ".xlsx"})
files

## 2. สำรวจข้อมูลดิบ
ดูว่าแต่ละปีใช้ชื่อคอลัมน์ตรงกันไหม และค่าที่ไม่เรียบร้อยมีอะไรบ้าง

In [ ]:
raws = {f.name: read_raw(f) for f in files}
pd.DataFrame({name: pd.Series(df.columns) for name, df in raws.items()})

In [ ]:
first = next(iter(raws.values()))
std = standardize_columns(first)
for col in ["incident_date", "incident_time", "province", "cause", "vehicle"]:
    if col in std:
        print(f"--- {col}: {std[col].nunique()} unique values")
        print(std[col].value_counts(dropna=False).head(15).to_string(), "\n")

## 3. ทำความสะอาด
ทุกกฎอยู่ใน `roadsafety/clean.py` และแต่ละขั้นจะบันทึกจำนวนแถวที่ถูกแก้ไว้ใน log

In [ ]:
frames, logs = [], []
for name, raw in raws.items():
    log = CleaningLog()
    frames.append(clean(raw, log))
    logs.append(log.to_frame().assign(file=name))
from roadsafety.clean import drop_duplicate_accidents
df = add_features(drop_duplicate_accidents(pd.concat(frames, ignore_index=True)).reset_index(drop=True))
pd.concat(logs)

In [ ]:
df[["date", "hour", "province_th", "province_en", "region", "cause_group", "vehicle_group",
    "fatalities", "injuries", "period", "festival_day"]].sample(10, random_state=1)

## 4. ตามเวลา

ก่อนเทียบเทศกาลกับวันปกติ ต้องดูก่อนว่าการ *บันทึก* ข้อมูลเหมือนกันไหม: จำนวนอุบัติเหตุกระโดด ~4 เท่าในวันแรกของแคมเปญพอดี แล้วตกกลับวันหลังจบ → เป็นผลจากการเก็บข้อมูลเข้มข้นช่วงแคมเปญ

In [ ]:
analysis.around_nominal_start(df).pivot(index="offset", columns="period", values="accidents").round(0)

In [ ]:
analysis.campaign_windows(df)

In [ ]:
analysis.agency_mix(df)

In [ ]:
analysis.period_rates(df)

In [ ]:
analysis.festival_by_year(df)

In [ ]:
analysis.festival_day_profile(df)

In [ ]:
hours = analysis.hour_profile(df)
hours.pivot(index="hour", columns="group", values="share_of_accidents").style.format("{:.1%}")

## 5. ตามพื้นที่
*Burden* = จังหวัดที่มีผู้เสียชีวิตช่วงแคมเปญมากที่สุด, *Severity* = ผู้เสียชีวิตต่ออุบัติเหตุ 100 ครั้ง (เกิดเหตุแล้วตายง่ายแค่ไหน)

In [ ]:
prov = analysis.province_table(df)
prov.head(15)

In [ ]:
prov.dropna(subset=['festival_deaths_per_100_accidents']).sort_values('festival_deaths_per_100_accidents', ascending=False).head(15)

In [ ]:
analysis.region_table(df)

## 6. สาเหตุและประเภทรถ

In [ ]:
analysis.category_comparison(df, 'cause_group')

In [ ]:
analysis.category_comparison(df, 'vehicle_group')

## 7. สร้างรายงานฉบับเต็ม (กราฟ + insight + ข้อเสนอแนะ)

In [ ]:
from roadsafety.report import build
path = build(files)
print(path)